# How Iso-Surface Extraction Works

Many shapes enter a program as a scalar field rather than a mesh: a
signed distance function, the output of a neural network, a CT scan,
the level set of a fluid simulation. The field gives a value at any
point, and the shape is an iso-surface: the set of points where the
value equals some level, usually zero. Most of the graphics stack,
though, consumes triangle meshes -- renderers, physics engines, 3D
printers, mesh-based losses. Iso-surface extraction converts one
representation into the other: sample the field on a grid, then
build a mesh of the level set.

The catch is that the field is only known at the grid corners.
Every extraction method reconstructs the surface from those samples,
and they all answer the same two questions: where do the mesh
vertices go, and how are they connected? The answers split the
methods into two families, primal and dual.


In [1]:
import torch
import isoext
from isoext import viewer
from isoext.sdf import get_sdf_normal, project_to_surface


## Where the Surface Crosses an Edge

Everything starts on the grid edges. If an edge's two endpoint
values straddle the level -- one above, one below -- the surface
crosses that edge exactly once, and linear interpolation says where:
with level-relative endpoint values `va` and `vb`, the crossing sits
at the fraction `t = va / (va - vb)` of the way from `a` to `b`. A
corner value near zero pulls the crossing toward that corner. Every
method on this page places geometry using these crossing points;
`isoext.get_intersection` computes them directly.

Below, a single cell with one corner inside (red). The surface
crosses the three edges that connect it to outside corners (blue),
and the gold dots are the interpolated crossings -- closer to the
red corner on the edges whose outside value is large:


In [2]:
cell = isoext.UniformGrid([2, 2, 2])
cell.set_values(torch.tensor(
    [-0.5, 0.3, 0.4, 0.5, 0.6, 0.4, 0.3, 0.2], device="cuda"
).reshape(2, 2, 2))

its = isoext.get_intersection(cell)
crossings = its.get_points()
print(f"{len(crossings)} crossings:")
for p in crossings.tolist():
    print("  ", [round(x, 2) for x in p])
viewer.embed(grid=cell, height=300,
             draw=lambda s: viewer.add_points(s, crossings, point_size=0.09))


3 crossings:
   [-1.0, -1.0, 0.25]
   [-1.0, 0.11, -1.0]
   [-0.09, -1.0, -1.0]


## Primal: Marching Cubes

{doc}`Marching cubes <marching_cubes>` visits every cell
independently. Each of a cell's 8 corners is either inside or
outside, giving 256 possible sign patterns, and a lookup table maps
each pattern to a set of triangles whose vertices are the edge
crossings. For the pattern above the table produces a single
triangle that cuts the inside corner off -- its three vertices are
exactly the three gold dots:


In [3]:
v, f = isoext.marching_cubes(cell)
print(f"{f.shape[0]} triangle(s)")
viewer.embed(v, f, color="steelblue", flat_shading=True, side="double", height=300,
             grid=cell, draw=lambda s: viewer.add_points(s, crossings, point_size=0.09))


1 triangle(s)


With more corners inside, the patterns get richer, and some of them
admit more than one valid triangulation. How the table variants
resolve those choices -- and what goes wrong when they do it badly
-- is the subject of {doc}`mc_variants`.

## Marching Tetrahedra: Simpler Cells, No Choices

{doc}`Marching tetrahedra <marching_tetrahedra>` runs the same
recipe on simpler cells. Each cube is split into six tetrahedra that
all share the cube's main diagonal. The purple lines below are the
extra edges the split introduces: one diagonal per face, plus the
main diagonal itself:


In [4]:
corners = cell.get_points().reshape(8, 3)
tets = [[0, 4, 6, 7], [0, 5, 4, 7], [0, 6, 2, 7],
        [0, 2, 3, 7], [0, 1, 5, 7], [0, 3, 1, 7]]  # the split used by src/mt.cu
edges = {tuple(sorted((t[i], t[j]))) for t in tets for i in range(4) for j in range(i + 1, 4)}
diagonals = torch.stack([torch.stack([corners[a], corners[b]])
                         for a, b in edges if bin(a ^ b).count("1") > 1])
print(f"{len(diagonals)} diagonal edges")
viewer.embed(grid=cell, height=300,
             draw=lambda s: viewer.add_lines(s, diagonals, color="orchid"))


7 diagonal edges


A tetrahedron has 4 corners and only 16 sign patterns, each
producing one or two triangles with no choices to make, so the mesh
is consistent by construction. The price shows on the same field as
before: the surface now also crosses the diagonals, and one triangle
becomes six:


In [5]:
v, f = isoext.marching_tetrahedra(cell)
print(f"{f.shape[0]} triangle(s)")
viewer.embed(v, f, color="mediumpurple", flat_shading=True, side="double", height=300,
             grid=cell, draw=lambda s: viewer.add_lines(s, diagonals, color="orchid"))


6 triangle(s)


## Dual: One Vertex per Cell, One Quad per Edge

The dual methods, {doc}`surface nets <surface_nets>` and
{doc}`dual contouring <dual_contouring>`, swap the roles of cells
and edges. Instead of putting vertices on the grid edges and
triangles inside each cell, they put one vertex *inside* each cell
the surface crosses. The connectivity comes from the crossed edges:
every crossed edge is surrounded by four cells, and their four
vertices are joined into a quad that straddles the edge.

Below, a flat field crosses every vertical edge of a 2x2x1 block of
cells (gold dots). Only the central edge has four cells around it,
so the mesh is exactly one quad -- two triangles -- connecting the
four cell vertices around that edge. The other crossings sit on
boundary edges with fewer neighbors and get no face:


In [6]:
block = isoext.UniformGrid([3, 3, 2], aabb_min=[-1, -1, -0.5], aabb_max=[1, 1, 0.5])
block.set_values(block.get_points()[..., 2])  # a flat surface at z = 0

v, f = isoext.surface_nets(block)
its = isoext.get_intersection(block)
print(f"{f.shape[0]} triangles")
viewer.embed(v, f, color="seagreen", flat_shading=True, side="double", height=300,
             grid=block,
             draw=lambda s: viewer.add_points(s, its.get_points(), point_size=0.07))


2 triangles


That leaves each cell one decision: where inside the cell does the
vertex go? The two methods answer differently.

- **Surface nets** places it at the centroid of the cell's edge
  crossings. Simple, smooth, and it needs nothing but the crossings.
- **Dual contouring** uses the surface normal at each crossing.
  Every crossing plus its normal defines a tangent plane, and the
  vertex is placed to minimize the total squared distance to all of
  those planes (a least-squares problem known as the **QEF**). If the
  cell contains a sharp corner, the planes are the corner's faces,
  and their intersection point *is* the corner. For this to be
  exact, the crossings are first refined from their interpolated
  estimates onto the true surface with `project_to_surface`, the
  same recipe {doc}`dual_contouring` uses for sharp features.

The cell below contains the corner of a tilted box, modeled as
the intersection of three half-spaces with rotated normals, meeting
at (0.2, 0.25, 0.3). The
gold dots are the crossings, the arrows their normals, and each
translucent square a piece of the tangent plane they define, drawn
large enough to reach their common intersection.
Surface nets puts the vertex at their centroid (green), inside the
box and away from the feature. The QEF solution (red) is the point
closest to all three planes -- here their exact intersection, the
corner itself at (0.2, 0.25, 0.3). (When the crossings do not pin down all three axes
-- flat faces, straight creases -- the QEF is degenerate, and real
implementations regularize it toward the centroid; see
{doc}`dual_contouring`.)


In [7]:
# Three tilted half-spaces whose faces meet at one point.
planes = torch.nn.functional.normalize(torch.tensor([
    [1.0, 0.3, -0.2],
    [-0.25, 1.0, 0.3],
    [0.2, -0.3, 1.0],
], device="cuda"), dim=-1)
corner = torch.tensor([0.2, 0.25, 0.3], device="cuda")
corner_box = lambda p: ((p - corner) @ planes.T).amax(dim=-1)

corner_cell = isoext.UniformGrid([2, 2, 2])
corner_cell.set_values(corner_box(corner_cell.get_points()))

its = isoext.get_intersection(corner_cell)
crossings = project_to_surface(corner_box, its.get_points())
normals = get_sdf_normal(corner_box, crossings)

centroid = crossings.mean(dim=0, keepdim=True)
# The QEF: minimize the squared distances to the tangent planes.
qef = torch.linalg.lstsq(normals, (normals * crossings).sum(-1)).solution[None]
print(f"surface nets vertex: {[round(x, 2) for x in centroid[0].tolist()]}")
print(f"dual contouring vertex: {[round(x, 2) for x in qef[0].tolist()]}")

# Draw each tangent plane spanning from its crossing to the intersection.
viewer.embed(grid=corner_cell, height=360, draw=lambda s: (
    viewer.add_points(s, crossings, point_size=0.09),
    viewer.add_arrows(s, crossings, 0.45 * normals),
    viewer.add_planes(s, (crossings + qef) / 2, normals, size=1.0, opacity=0.3),
    viewer.add_points(s, centroid, color="seagreen", point_size=0.13),
    viewer.add_points(s, qef, color="crimson", point_size=0.13),
))


surface nets vertex: [-0.57, -0.55, -0.61]
dual contouring vertex: [0.2, 0.25, 0.3]


Zooming out one step, the two families are easy to tell apart. The
smallest possible sphere -- one inside sample surrounded by outside
samples on 2x2x2 cells -- comes out as an octahedron under marching
cubes, with its 6 vertices on the grid edges, and as a cube under
surface nets, with its 8 vertices inside the cells:


In [8]:
tiny = isoext.UniformGrid([3, 3, 3])
tiny.set_values(tiny.get_points().norm(dim=-1) - 0.7)

v, f = isoext.marching_cubes(tiny)
print(f"marching_cubes: {v.shape[0]} vertices, {f.shape[0]} triangles")
viewer.embed(v, f, wireframe=True, color="orange", height=320, grid=tiny)


marching_cubes: 6 vertices, 8 triangles


In [9]:
v, f = isoext.surface_nets(tiny)
print(f"surface_nets:   {v.shape[0]} vertices, {f.shape[0]} triangles")
viewer.embed(v, f, wireframe=True, color="seagreen", height=320, grid=tiny)


surface_nets:   8 vertices, 12 triangles


## How the Methods Relate

| Method | Vertices | Faces | Ambiguities |
|---|---|---|---|
| Marching cubes | on grid edges | per cell, from a table | resolved by the variant |
| Marching tetrahedra | on tetrahedron edges | per tetrahedron | none |
| Surface nets | one per cell, at the centroid | one quad per crossed edge | none |
| Dual contouring | one per cell, from the QEF | one quad per crossed edge | none |

All four run on the same grids, share the same edge crossings, and
answer the same two questions from the top of the page. The primal
methods answer with vertices at the crossings themselves, so the
mesh interpolates the samples exactly. The dual methods answer with
faces there instead, which tends to give better-shaped triangles
and, with good normals, sharp features -- at the cost of vertex
positions that are estimated rather than interpolated.

{doc}`performance` compares their speed, and the papers behind every
method are collected in {doc}`references`.
